# Lab 4 — CNN для Colab

Этот notebook повторяет логику файлов `config.py`, `data.py`, `cnn.py`, `block1_main.py` и `block2_main.py` из `lab4`.

Порядок работы:
1. Выполнить ячейки сверху вниз.
2. Для основного перебора гиперпараметров запустить ячейку **Run Block 1**.
3. Для сравнения оптимизаторов и демонстрации переобучения запустить ячейку **Run Block 2**.

In [ ]:
import copy
import itertools
import math
import os
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

SEED = 52

# ── Датасет ───────────────────────────────────────────────────────
DATASET_NAME = os.environ.get('SMGMO4_DATASET', 'MNIST')
DATA_ROOT = os.environ.get('SMGMO4_DATA_ROOT', './data')

# ── Размеры выборок ───────────────────────────────────────────────
TRAIN_SUBSET = int(os.environ.get('SMGMO4_TRAIN_SUBSET', '10000'))
TEST_SUBSET = int(os.environ.get('SMGMO4_TEST_SUBSET', '2000'))
VAL_RATIO = float(os.environ.get('SMGMO4_VAL_RATIO', '0.2'))

# ── Обучение ──────────────────────────────────────────────────────
BATCH_SIZE = int(os.environ.get('SMGMO4_BATCH_SIZE', '128'))
N_EPOCHS = int(os.environ.get('SMGMO4_N_EPOCHS', '20'))
PATIENCE = int(os.environ.get('SMGMO4_PATIENCE', '5'))
MIN_DELTA = float(os.environ.get('SMGMO4_MIN_DELTA', '1e-4'))
TARGET_ACC = float(os.environ.get('SMGMO4_TARGET_ACC', '0.90'))
FINAL_EPOCHS = int(os.environ.get('SMGMO4_FINAL_EPOCHS', str(N_EPOCHS)))

# ── DataLoader / preprocessing ────────────────────────────────────
NORMALIZE_IMAGES = os.environ.get('SMGMO4_NORMALIZE_IMAGES', '1') == '1'
NUM_WORKERS = int(os.environ.get('SMGMO4_NUM_WORKERS', '2'))
_pin_memory_raw = os.environ.get('SMGMO4_PIN_MEMORY', 'auto').strip().lower()
PIN_MEMORY = None if _pin_memory_raw == 'auto' else _pin_memory_raw in ('1', 'true', 'yes', 'on')

# ── Демонстрация переобучения ─────────────────────────────────────
OVERFIT_TRAIN = int(os.environ.get('SMGMO4_OVERFIT_TRAIN', '300'))
OVERFIT_EPOCHS = int(os.environ.get('SMGMO4_OVERFIT_EPOCHS', '120'))

# ── Визуализация ──────────────────────────────────────────────────
DPI = 130
plt.rcParams['figure.dpi'] = DPI

print({
    'dataset': DATASET_NAME,
    'train_subset': TRAIN_SUBSET,
    'test_subset': TEST_SUBSET,
    'batch_size': BATCH_SIZE,
    'n_epochs': N_EPOCHS,
    'target_acc': TARGET_ACC,
})

In [ ]:
DATASET_INFO = {
    'MNIST': {
        'loader': datasets.MNIST,
        'classes': [str(i) for i in range(10)],
        'in_channels': 1,
        'image_size': 28,
    },
    'FashionMNIST': {
        'loader': datasets.FashionMNIST,
        'classes': [
            'T-shirt', 'Trouser', 'Pullover', 'Dress', 'Coat',
            'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Boot',
        ],
        'in_channels': 1,
        'image_size': 28,
    },
}

DATASET_NORMALIZATION = {
    'MNIST': {
        'mean': (0.1307,),
        'std': (0.3081,),
    },
    'FashionMNIST': {
        'mean': (0.2860,),
        'std': (0.3530,),
    },
}

def load_image_dataset(name, data_root):
    if name not in DATASET_INFO:
        raise ValueError(f'Неизвестный датасет: {name}')
    info = DATASET_INFO[name]
    tfms = [transforms.ToTensor()]
    if NORMALIZE_IMAGES and name in DATASET_NORMALIZATION:
        stats = DATASET_NORMALIZATION[name]
        tfms.append(transforms.Normalize(stats['mean'], stats['std']))
    transform = transforms.Compose(tfms)
    train = info['loader'](root=data_root, train=True, download=True, transform=transform)
    test = info['loader'](root=data_root, train=False, download=True, transform=transform)
    return train, test, info

def take_subset(dataset, subset_size, seed=SEED):
    rng = np.random.default_rng(seed)
    n = min(subset_size, len(dataset))
    idx = rng.choice(len(dataset), size=n, replace=False)
    return Subset(dataset, idx.tolist())

def split_train_val(dataset, val_ratio, seed=SEED):
    rng = np.random.default_rng(seed)
    idx = np.arange(len(dataset))
    rng.shuffle(idx)
    n_val = int(len(dataset) * val_ratio)
    n_train = len(dataset) - n_val
    return (Subset(dataset, idx[:n_train].tolist()), Subset(dataset, idx[n_train:].tolist()))

def make_loaders(train_ds, val_ds, test_ds, batch_size, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY):
    use_pin_memory = torch.cuda.is_available() if pin_memory is None else (pin_memory and torch.cuda.is_available())
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=use_pin_memory)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=use_pin_memory)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=use_pin_memory)
    return train_loader, val_loader, test_loader

def plot_random_images(dataset, class_names, title, rows=4, cols=4, seed=SEED):
    rng = np.random.default_rng(seed)
    _, axes = plt.subplots(rows, cols, figsize=(2.2 * cols, 2.2 * rows))
    plt.suptitle(title, fontsize=13)
    for ax in axes.ravel():
        i = int(rng.integers(0, len(dataset)))
        img, label = dataset[i]
        ax.imshow(img.squeeze(0), cmap='gray')
        ax.set_title(class_names[int(label)], fontsize=9)
        ax.axis('off')
    plt.tight_layout()
    plt.show()
    plt.close()

In [ ]:
def get_device():
    if torch.cuda.is_available():
        return torch.device('cuda')
    if torch.backends.mps.is_available() and torch.backends.mps.is_built():
        return torch.device('mps')
    return torch.device('cpu')

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

class ConvBlockA(nn.Module):
    """Conv2d -> ReLU -> MaxPool."""
    def __init__(self, in_ch, out_ch, conv_k, conv_s, pool_k, pool_s):
        super().__init__()
        pad = conv_k // 2
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=conv_k, stride=conv_s, padding=pad),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=pool_k, stride=pool_s),
        )

    def forward(self, x):
        return self.block(x)

class ConvBlockB(nn.Module):
    """Conv2d -> ReLU -> Conv2d -> ReLU -> MaxPool."""
    def __init__(self, in_ch, out_ch, conv_k, conv_s, pool_k, pool_s):
        super().__init__()
        pad = conv_k // 2
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=conv_k, stride=conv_s, padding=pad),
            nn.ReLU(),
            nn.Conv2d(out_ch, out_ch, kernel_size=conv_k, stride=1, padding=pad),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=pool_k, stride=pool_s),
        )

    def forward(self, x):
        return self.block(x)

class ImageCNN(nn.Module):
    """Последовательность свёрточных блоков -> AdaptiveAvgPool -> Linear."""
    def __init__(self, in_channels, num_classes, block_type, channels, conv_k, conv_s, pool_k, pool_s, dropout_rate=0.0):
        super().__init__()
        if block_type not in ('a', 'b'):
            raise ValueError("block_type должен быть 'a' или 'b'")
        Block = ConvBlockA if block_type == 'a' else ConvBlockB

        layers = []
        prev = in_channels
        for ch in channels:
            layers.append(Block(prev, ch, conv_k, conv_s, pool_k, pool_s))
            prev = ch
        self.features = nn.Sequential(*layers)

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Dropout(dropout_rate),
            nn.Linear(prev, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

def is_valid_config(image_size, num_blocks, conv_k, conv_s, pool_k, pool_s):
    s = image_size
    pad = conv_k // 2
    for _ in range(num_blocks):
        s = math.floor((s + 2 * pad - conv_k) / conv_s + 1)
        if s < pool_k:
            return False
        s = math.floor((s - pool_k) / pool_s + 1)
        if s < 1:
            return False
    return True

def make_optimizer(name, params, lr):
    if name == 'Adam':
        return torch.optim.Adam(params, lr=lr)
    if name == 'AdamW':
        return torch.optim.AdamW(params, lr=lr, weight_decay=1e-4)
    if name == 'SGD':
        return torch.optim.SGD(params, lr=lr, momentum=0.9, nesterov=True)
    raise ValueError(f'Неизвестный оптимизатор: {name}')

def confusion_matrix(y_true, y_pred, n_classes):
    cm = np.zeros((n_classes, n_classes), dtype=int)
    for t, p in zip(y_true, y_pred):
        cm[int(t), int(p)] += 1
    return cm

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    total_correct = 0
    total = 0
    for x, y in loader:
        x = x.to(device)
        y = y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * y.size(0)
        total_correct += (out.argmax(dim=1) == y).sum().item()
        total += y.size(0)
    return total_loss / total, total_correct / total

@torch.no_grad()
def evaluate(model, loader, criterion, device, return_preds=False):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total = 0
    y_true_all, y_pred_all = [], []
    for x, y in loader:
        x = x.to(device)
        y = y.to(device)
        out = model(x)
        loss = criterion(out, y)
        preds = out.argmax(dim=1)
        total_loss += loss.item() * y.size(0)
        total_correct += (preds == y).sum().item()
        total += y.size(0)
        if return_preds:
            y_true_all.extend(y.cpu().numpy())
            y_pred_all.extend(preds.cpu().numpy())
    avg_loss = total_loss / total
    avg_acc = total_correct / total
    if return_preds:
        return avg_loss, avg_acc, np.array(y_true_all), np.array(y_pred_all)
    return avg_loss, avg_acc

def train_model(model, train_loader, val_loader, optimizer, criterion, device, n_epochs, target_acc, patience, min_delta, log_every=5, tag=''):
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    best_val_acc = -1.0
    best_state = copy.deepcopy(model.state_dict())
    best_epoch = 0
    epoch_to_target = None
    bad = 0

    t0 = time.perf_counter()
    for epoch in range(1, n_epochs + 1):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc = evaluate(model, val_loader, criterion, device)

        history['train_loss'].append(tr_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(tr_acc)
        history['val_acc'].append(val_acc)

        if epoch == 1 or epoch % log_every == 0 or epoch == n_epochs:
            prefix = f'[{tag}] ' if tag else ''
            print(f'  {prefix}эпоха {epoch:>3}/{n_epochs} | train_loss={tr_loss:.4f} val_loss={val_loss:.4f} | train_acc={tr_acc:.4f} val_acc={val_acc:.4f}')

        if epoch_to_target is None and val_acc >= target_acc:
            epoch_to_target = epoch

        if val_acc > best_val_acc + min_delta:
            best_val_acc = val_acc
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            bad = 0
        else:
            bad += 1

        if bad >= patience:
            print(f'  -> ранняя остановка на эпохе {epoch}')
            break

    model.load_state_dict(best_state)
    duration = time.perf_counter() - t0

    return {
        'history': history,
        'best_val_acc': best_val_acc,
        'best_epoch': best_epoch,
        'epoch_to_target': epoch_to_target,
        'duration_sec': duration,
        'trained_epochs': len(history['train_loss']),
    }

In [ ]:
BLOCK_TYPES = ['a', 'b']
CHANNELS_OPTIONS = [(16, 32), (32, 64)]
CONV_KERNELS = [3, 5]
CONV_STRIDES = [1]
POOL_KERNELS = [2]
POOL_STRIDES = [1, 2]
LEARNING_RATES = [1e-3]

def build_grid(image_size=28):
    grid = []
    cid = 1
    for bt, ch, ck, cs, pk, ps, lr in itertools.product(
        BLOCK_TYPES, CHANNELS_OPTIONS, CONV_KERNELS, CONV_STRIDES, POOL_KERNELS, POOL_STRIDES, LEARNING_RATES,
    ):
        if not is_valid_config(image_size, len(ch), ck, cs, pk, ps):
            continue
        grid.append({
            'id': cid,
            'block_type': bt, 'channels': ch,
            'conv_k': ck, 'conv_s': cs,
            'pool_k': pk, 'pool_s': ps,
            'lr': lr,
        })
        cid += 1
    return grid

def short_name(cfg):
    return f"#{cfg['id']:02d} blk{cfg['block_type']} ch={cfg['channels']} ck={cfg['conv_k']} ps={cfg['pool_s']}"

def cfg_pretty(cfg):
    return (
        f"block={cfg['block_type']} | channels={cfg['channels']} | "
        f"conv(k={cfg['conv_k']}, s={cfg['conv_s']}) | "
        f"pool(k={cfg['pool_k']}, s={cfg['pool_s']}) | "
        f"lr={cfg['lr']}"
    )

def run_one(cfg, train_loader, val_loader, info, device):
    set_seed(SEED)
    model = ImageCNN(
        in_channels=info['in_channels'],
        num_classes=len(info['classes']),
        block_type=cfg['block_type'],
        channels=cfg['channels'],
        conv_k=cfg['conv_k'], conv_s=cfg['conv_s'],
        pool_k=cfg['pool_k'], pool_s=cfg['pool_s'],
    ).to(device)
    optimizer = make_optimizer('Adam', model.parameters(), cfg['lr'])
    criterion = nn.CrossEntropyLoss()
    n_params = sum(p.numel() for p in model.parameters())
    print(f"\n>>> {cfg_pretty(cfg)} | params={n_params:,}")
    res = train_model(
        model, train_loader, val_loader, optimizer, criterion, device,
        n_epochs=N_EPOCHS, target_acc=TARGET_ACC, patience=PATIENCE,
        min_delta=MIN_DELTA, log_every=2, tag=f"#{cfg['id']:02d}",
    )
    res['cfg'] = cfg
    res['model'] = model
    res['n_params'] = n_params
    return res

def train_final_on_full(best_cfg, train_full_ds, test_ds, info, device, n_epochs=FINAL_EPOCHS):
    set_seed(SEED)
    model = ImageCNN(
        in_channels=info['in_channels'],
        num_classes=len(info['classes']),
        block_type=best_cfg['block_type'],
        channels=best_cfg['channels'],
        conv_k=best_cfg['conv_k'], conv_s=best_cfg['conv_s'],
        pool_k=best_cfg['pool_k'], pool_s=best_cfg['pool_s'],
    ).to(device)

    optimizer = make_optimizer('Adam', model.parameters(), best_cfg['lr'])
    criterion = nn.CrossEntropyLoss()

    use_pin_memory = torch.cuda.is_available() if PIN_MEMORY is None else (PIN_MEMORY and torch.cuda.is_available())
    train_loader_full = DataLoader(train_full_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=use_pin_memory)
    test_loader_full = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=use_pin_memory)

    history = {'train_loss': [], 'train_acc': []}
    t0 = time.perf_counter()
    print('\n' + '#' * 80)
    print(' ФИНАЛЬНОЕ ПЕРЕОБУЧЕНИЕ НА ПОЛНОМ TRAIN')
    print('#' * 80)
    print(f'  Конфиг: {cfg_pretty(best_cfg)}')
    print(f'  train size: {len(train_full_ds)} | test size: {len(test_ds)}')

    for epoch in range(1, n_epochs + 1):
        tr_loss, tr_acc = train_one_epoch(model, train_loader_full, criterion, optimizer, device)
        history['train_loss'].append(tr_loss)
        history['train_acc'].append(tr_acc)
        if epoch == 1 or epoch % 2 == 0 or epoch == n_epochs:
            print(f'  [final] эпоха {epoch:>3}/{n_epochs} | train_loss={tr_loss:.4f} train_acc={tr_acc:.4f}')

    test_loss, test_acc, y_true, y_pred = evaluate(model, test_loader_full, criterion, device, return_preds=True)
    duration = time.perf_counter() - t0
    return {
        'model': model,
        'history': history,
        'test_loss': test_loss,
        'test_acc': test_acc,
        'y_true': y_true,
        'y_pred': y_pred,
        'duration_sec': duration,
        'trained_epochs': n_epochs,
    }

def print_results_table(results):
    print('\n' + '=' * 110)
    print(f"{'#':>3} | {'blk':>3} | {'channels':>10} | {'ck':>2} | {'cs':>2} | {'pk':>2} | {'ps':>2} | {'lr':>7} | {'best_val':>9} | {'ep@90':>5} | {'epochs':>6} | {'time,s':>7}")
    print('-' * 110)
    for r in sorted(results, key=lambda x: -x['best_val_acc']):
        c = r['cfg']
        ep90 = r['epoch_to_target'] if r['epoch_to_target'] else '—'
        print(f"{c['id']:>3} | {c['block_type']:>3} | {str(c['channels']):>10} | {c['conv_k']:>2} | {c['conv_s']:>2} | {c['pool_k']:>2} | {c['pool_s']:>2} | {c['lr']:>7.4f} | {r['best_val_acc']:>9.4f} | {str(ep90):>5} | {r['trained_epochs']:>6} | {r['duration_sec']:>7.1f}")
    print('=' * 110)

def plot_curves(history, title, target_acc=TARGET_ACC, best_epoch=None):
    _, (ax_l, ax_a) = plt.subplots(1, 2, figsize=(13, 4.2))

    ax_l.plot(history['train_loss'], label='train loss')
    ax_l.plot(history['val_loss'], label='val loss')
    if best_epoch is not None:
        ax_l.axvline(best_epoch - 1, color='crimson', ls='--', label='best epoch')
    ax_l.set_title(f'{title} — loss')
    ax_l.set_xlabel('epoch')
    ax_l.set_ylabel('loss')
    ax_l.legend()
    ax_l.grid(True, alpha=0.3)

    ax_a.plot(history['train_acc'], label='train acc')
    ax_a.plot(history['val_acc'], label='val acc')
    ax_a.axhline(target_acc, color='green', ls='--', label=f'target {target_acc:.2f}')
    if best_epoch is not None:
        ax_a.axvline(best_epoch - 1, color='crimson', ls='--', label='best epoch')
    ax_a.set_title(f'{title} — accuracy')
    ax_a.set_xlabel('epoch')
    ax_a.set_ylabel('accuracy')
    ax_a.legend()
    ax_a.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
    plt.close()

def plot_confusion(cm, class_names, title):
    n = len(class_names)
    _, ax = plt.subplots(figsize=(8, 7))
    im = ax.imshow(cm, cmap='Blues')
    ax.set_title(title)
    ax.set_xticks(range(n))
    ax.set_yticks(range(n))
    ax.set_xticklabels(class_names, rotation=45, ha='right')
    ax.set_yticklabels(class_names)
    ax.set_xlabel('predicted')
    ax.set_ylabel('true')
    thr = cm.max() / 2 if cm.size else 0
    for i in range(n):
        for j in range(n):
            color = 'white' if cm[i, j] > thr else 'black'
            ax.text(j, i, str(cm[i, j]), ha='center', va='center', fontsize=8, color=color)
    plt.colorbar(im, ax=ax, fraction=0.046)
    plt.tight_layout()
    plt.show()
    plt.close()

def plot_param_compare(results, param_name, getter, target_acc=TARGET_ACC):
    groups = {}
    for r in results:
        groups.setdefault(getter(r['cfg']), []).append(r['best_val_acc'])
    keys = sorted(groups, key=str)
    means = [float(np.mean(groups[k])) for k in keys]
    stds = [float(np.std(groups[k])) for k in keys]

    _, ax = plt.subplots(figsize=(7, 4))
    x = np.arange(len(keys))
    ax.bar(x, means, yerr=stds, capsize=4, color='seagreen')
    ax.axhline(target_acc, color='red', ls='--', label=f'target {target_acc:.2f}')
    ax.set_title(f'Влияние «{param_name}» на val_acc (mean ± std)')
    ax.set_xticks(x)
    ax.set_xticklabels([str(k) for k in keys])
    ax.set_ylabel('val_acc')
    ax.set_ylim(min(means) * 0.95 if means else 0, 1.0)
    ax.legend()
    plt.tight_layout()
    plt.show()
    plt.close()

def plot_grid_summary(results, target_acc=TARGET_ACC):
    sorted_res = sorted(results, key=lambda r: -r['best_val_acc'])
    labels = [short_name(r['cfg']) for r in sorted_res]
    accs = [r['best_val_acc'] for r in sorted_res]
    eps90 = [r['epoch_to_target'] or 0 for r in sorted_res]

    _, axes = plt.subplots(2, 1, figsize=(max(10, len(labels) * 0.55), 8))
    x = np.arange(len(labels))

    bars = axes[0].bar(x, accs, color='steelblue')
    axes[0].axhline(target_acc, color='red', ls='--', label=f'target {target_acc:.2f}')
    axes[0].set_title('Best val_acc по экспериментам (по убыванию)')
    axes[0].set_ylim(min(accs) * 0.95 if accs else 0, 1.0)
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(labels, rotation=70, ha='right', fontsize=7)
    axes[0].legend()
    for bar, val in zip(bars, accs):
        axes[0].text(bar.get_x() + bar.get_width() / 2, val + 0.002, f'{val:.3f}', ha='center', fontsize=6)

    axes[1].bar(x, eps90, color='darkorange')
    axes[1].set_title(f'Эпоха достижения val_acc >= {target_acc:.2f} (0 = не достигнут)')
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(labels, rotation=70, ha='right', fontsize=7)

    plt.tight_layout()
    plt.show()
    plt.close()

def main_block1():
    set_seed(SEED)
    device = get_device()
    print('=' * 80)
    print(f' Задание 4 — Блок 1: перебор гиперпараметров CNN на {DATASET_NAME}')
    print(f' Устройство: {device}')
    print('=' * 80)

    train_full, test_full, info = load_image_dataset(DATASET_NAME, DATA_ROOT)

    train_grid = take_subset(train_full, TRAIN_SUBSET, seed=SEED)
    train_ds, val_ds = split_train_val(train_grid, VAL_RATIO, seed=SEED)
    test_ds = take_subset(test_full, TEST_SUBSET, seed=SEED + 1)

    print(f' train: {len(train_ds)} | val: {len(val_ds)} | test: {len(test_ds)}')
    train_loader, val_loader, test_loader = make_loaders(train_ds, val_ds, test_ds, batch_size=BATCH_SIZE)

    plot_random_images(train_ds, info['classes'], f'{DATASET_NAME}: примеры из train')

    grid = build_grid(image_size=info['image_size'])
    print(f'\nКонфигураций в сетке: {len(grid)}')

    results = []
    for cfg in grid:
        results.append(run_one(cfg, train_loader, val_loader, info, device))

    print_results_table(results)

    best = max(results, key=lambda r: r['best_val_acc'])
    print('\n' + '#' * 80)
    print(' ЛУЧШАЯ КОНФИГУРАЦИЯ')
    print('#' * 80)
    print(f"  {cfg_pretty(best['cfg'])}")
    print(f"  best val_acc:        {best['best_val_acc']:.4f}")
    print(f"  лучшая эпоха:        {best['best_epoch']}")
    print(f"  эпох до 90%:         {best['epoch_to_target']}")
    print(f"  параметров:          {best['n_params']:,}")
    print(f"  время обучения:      {best['duration_sec']:.1f} сек")

    test_loss, test_acc, _, _ = evaluate(best['model'], test_loader, nn.CrossEntropyLoss(), device, return_preds=True)
    print(f'\n  test loss:           {test_loss:.4f}')
    print(f'  test accuracy:       {test_acc:.4f}')

    final_res = train_final_on_full(best['cfg'], train_full, test_full, info, device)
    print('\n' + '=' * 80)
    print(' Финальные метрики после переобучения на полном train')
    print('=' * 80)
    print(f"  epochs:              {final_res['trained_epochs']}")
    print(f"  train time:          {final_res['duration_sec']:.1f} сек")
    print(f"  final test loss:     {final_res['test_loss']:.4f}")
    print(f"  final test accuracy: {final_res['test_acc']:.4f}")

    plot_grid_summary(results)
    plot_curves(best['history'], title=short_name(best['cfg']), best_epoch=best['best_epoch'])

    cm = confusion_matrix(final_res['y_true'], final_res['y_pred'], len(info['classes']))
    plot_confusion(cm, info['classes'], f'Confusion matrix — final full-train ({DATASET_NAME})')

    plot_param_compare(results, 'block_type', lambda c: c['block_type'])
    plot_param_compare(results, 'channels', lambda c: str(c['channels']))
    plot_param_compare(results, 'conv_kernel_size', lambda c: c['conv_k'])
    plot_param_compare(results, 'pool_stride', lambda c: c['pool_s'])

    reached = [r for r in results if r['epoch_to_target'] is not None]
    print('\n' + '=' * 80)
    print(' Оценка числа эпох до accuracy >= 90%')
    print('=' * 80)
    if reached:
        eps = [r['epoch_to_target'] for r in reached]
        print(f'  достигли цели:           {len(reached)}/{len(results)}')
        print(f'  минимум эпох:            {min(eps)}')
        print(f'  медиана:                 {int(np.median(eps))}')
        print(f"  у лучшей конфигурации:   {best['epoch_to_target']}")
    else:
        print('  Ни одна конфигурация не достигла 90% — увеличьте N_EPOCHS.')

    return {'results': results, 'best': best, 'final_res': final_res, 'info': info}

In [ ]:
BASE_CFG = {
    'block_type': 'b',
    'channels': (32, 64),
    'conv_k': 3,
    'conv_s': 1,
    'pool_k': 2,
    'pool_s': 2,
}

OPT_VARIANTS = [
    ('Adam', 1e-3),
    ('Adam', 3e-3),
    ('AdamW', 1e-3),
    ('SGD', 1e-2),
    ('SGD', 5e-2),
]

def make_model(info, device, dropout=0.0):
    return ImageCNN(
        in_channels=info['in_channels'],
        num_classes=len(info['classes']),
        block_type=BASE_CFG['block_type'],
        channels=BASE_CFG['channels'],
        conv_k=BASE_CFG['conv_k'], conv_s=BASE_CFG['conv_s'],
        pool_k=BASE_CFG['pool_k'], pool_s=BASE_CFG['pool_s'],
        dropout_rate=dropout,
    ).to(device)

def study_optimizers(train_loader, val_loader, info, device, n_epochs=10, patience=4):
    print('=' * 80)
    print(' Сравнение оптимизаторов и learning rate')
    print('=' * 80)
    print(f" Базовая архитектура: block={BASE_CFG['block_type']}, channels={BASE_CFG['channels']}, conv_k={BASE_CFG['conv_k']}")

    results = []
    criterion = nn.CrossEntropyLoss()
    for opt_name, lr in OPT_VARIANTS:
        set_seed(SEED)
        model = make_model(info, device)
        optimizer = make_optimizer(opt_name, model.parameters(), lr)
        print(f'\n--- {opt_name} | lr={lr} ---')
        res = train_model(
            model, train_loader, val_loader, optimizer, criterion, device,
            n_epochs=n_epochs, target_acc=TARGET_ACC, patience=patience,
            min_delta=1e-4, log_every=1, tag=f'{opt_name}-{lr}',
        )
        res['opt_name'] = opt_name
        res['lr'] = lr
        results.append(res)

    print('\nИтог сравнения:')
    print(f"  {'optimizer':>9} | {'lr':>7} | {'best_val':>9} | {'ep@90':>5}")
    print('  ' + '-' * 38)
    for r in results:
        ep = r['epoch_to_target'] if r['epoch_to_target'] else '—'
        print(f"  {r['opt_name']:>9} | {r['lr']:>7.4f} | {r['best_val_acc']:>9.4f} | {str(ep):>5}")
    return results

def plot_optimizer_curves(results, target_acc=TARGET_ACC):
    _, ax = plt.subplots(figsize=(9, 5))
    for r in results:
        epochs = np.arange(1, len(r['history']['val_acc']) + 1)
        ax.plot(epochs, r['history']['val_acc'], 'o-', label=f"{r['opt_name']} lr={r['lr']}")
    ax.axhline(target_acc, color='red', ls='--', label=f'target {target_acc:.2f}')
    ax.set_title('Сравнение оптимизаторов и learning rate (val accuracy)')
    ax.set_xlabel('epoch')
    ax.set_ylabel('val accuracy')
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.tight_layout()
    plt.show()
    plt.close()

def plot_overfit_compare(res_no, res_dr):
    _, axes = plt.subplots(1, 2, figsize=(14, 4.5))

    h1, h2 = res_no['history'], res_dr['history']
    axes[0].plot(h1['train_loss'], label='train (no-reg)')
    axes[0].plot(h1['val_loss'], label='val (no-reg)')
    axes[0].plot(h2['train_loss'], label='train (dropout)', ls='--')
    axes[0].plot(h2['val_loss'], label='val (dropout)', ls='--')
    axes[0].set_title('Loss: переобучение vs Dropout')
    axes[0].set_xlabel('epoch')
    axes[0].set_ylabel('loss')
    axes[0].grid(True, alpha=0.3)
    axes[0].legend()

    axes[1].plot(h1['train_acc'], label='train (no-reg)')
    axes[1].plot(h1['val_acc'], label='val (no-reg)')
    axes[1].plot(h2['train_acc'], label='train (dropout)', ls='--')
    axes[1].plot(h2['val_acc'], label='val (dropout)', ls='--')
    axes[1].set_title('Accuracy: переобучение vs Dropout')
    axes[1].set_xlabel('epoch')
    axes[1].set_ylabel('accuracy')
    axes[1].grid(True, alpha=0.3)
    axes[1].legend()

    plt.tight_layout()
    plt.show()
    plt.close()

def run_overfit_demo(info, device):
    print('\n' + '=' * 80)
    print(' Демонстрация переобучения на малой подвыборке')
    print('=' * 80)

    train_full, test_full, _ = load_image_dataset(DATASET_NAME, DATA_ROOT)
    train_small = take_subset(train_full, OVERFIT_TRAIN, seed=SEED)
    train_ds, val_ds = split_train_val(train_small, VAL_RATIO, seed=SEED)
    test_ds = take_subset(test_full, TEST_SUBSET, seed=SEED + 1)

    train_loader, val_loader, _ = make_loaders(train_ds, val_ds, test_ds, batch_size=BATCH_SIZE)

    print(f' train: {len(train_ds)} | val: {len(val_ds)}')

    criterion = nn.CrossEntropyLoss()

    print('\n--- без регуляризации (dropout=0) ---')
    set_seed(SEED)
    model_no = make_model(info, device, dropout=0.0)
    optimizer = make_optimizer('Adam', model_no.parameters(), 1e-3)
    res_no = train_model(
        model_no, train_loader, val_loader, optimizer, criterion, device,
        n_epochs=OVERFIT_EPOCHS, target_acc=1.0, patience=OVERFIT_EPOCHS,
        min_delta=0.0, log_every=20, tag='no-reg',
    )

    print('\n--- с Dropout=0.4 ---')
    set_seed(SEED)
    model_dr = make_model(info, device, dropout=0.4)
    optimizer = make_optimizer('Adam', model_dr.parameters(), 1e-3)
    res_dr = train_model(
        model_dr, train_loader, val_loader, optimizer, criterion, device,
        n_epochs=OVERFIT_EPOCHS, target_acc=1.0, patience=OVERFIT_EPOCHS,
        min_delta=0.0, log_every=20, tag='dropout',
    )

    plot_overfit_compare(res_no, res_dr)

    print('\nИтог переобучения:')
    for tag, r in [('без рег.', res_no), ('dropout 0.4', res_dr)]:
        h = r['history']
        gap = h['train_acc'][-1] - h['val_acc'][-1]
        print(f"  {tag:<14} | train_acc={h['train_acc'][-1]:.4f} | val_acc={h['val_acc'][-1]:.4f} | gap={gap:+.4f}")

    return {'no_reg': res_no, 'dropout': res_dr}

def main_block2():
    set_seed(SEED)
    device = get_device()
    print('=' * 80)
    print(f' Задание 4 — Блок 2: оптимизаторы и переобучение ({DATASET_NAME})')
    print(f' Устройство: {device}')
    print('=' * 80)

    train_full, test_full, info = load_image_dataset(DATASET_NAME, DATA_ROOT)
    train_full = take_subset(train_full, TRAIN_SUBSET, seed=SEED)
    train_ds, val_ds = split_train_val(train_full, VAL_RATIO, seed=SEED)
    test_ds = take_subset(test_full, TEST_SUBSET, seed=SEED + 1)
    train_loader, val_loader, _ = make_loaders(train_ds, val_ds, test_ds, batch_size=BATCH_SIZE)

    opt_results = study_optimizers(train_loader, val_loader, info, device, n_epochs=min(N_EPOCHS, 10))
    plot_optimizer_curves(opt_results)

    overfit_results = run_overfit_demo(info, device)
    return {'opt_results': opt_results, 'overfit_results': overfit_results, 'info': info}

## Run Block 1
Запускает перебор гиперпараметров, выбирает лучший конфиг, дообучает его на полном train и строит графики.

In [ ]:
block1_results = main_block1()

## Run Block 2
Запускает сравнение оптимизаторов и learning rate на базовой архитектуре, затем демонстрацию переобучения.

In [ ]:
block2_results = main_block2()